# Diabetes Risk Prediction — Model Interpretation with SHAP

**Goal:** Explain what drives the final XGBoost model's predictions, both globally
(across the whole dataset) and for individual patients.

Sections:
1. Recreate data split and retrain final model
2. Compute SHAP values
3. Global feature importance (summary plot)
4. Feature effect direction (beeswarm plot)
5. Individual prediction explanation (waterfall plot)
6. Dependence plots for top features
7. Key takeaways

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

RANDOM_STATE = 42
shap.initjs()

## 1. Recreate Data Split & Retrain Final Model

Same split as notebooks 02/03, so this analysis lines up with your reported results.
If you saved the model in notebook 03 (`joblib.dump`), you can load it instead of retraining.

In [ ]:
from ucimlrepo import fetch_ucirepo

cdc_diabetes = fetch_ucirepo(id=891)
X_raw = cdc_diabetes.data.features
y_raw = cdc_diabetes.data.targets

df = pd.concat([X_raw, y_raw], axis=1)
df = df.drop_duplicates().reset_index(drop=True)

target_col = 'Diabetes_binary'
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

print('Model retrained. Test shape:', X_test.shape)

## 2. Compute SHAP Values

`TreeExplainer` is fast and exact for tree-based models like XGBoost. We use a sample of
the test set for the summary/beeswarm plots to keep things quick — SHAP computation scales
with dataset size.

In [ ]:
# Sample for speed — increase sample_size if you want more precision and have time to spare
sample_size = 2000
X_sample = X_test.sample(n=min(sample_size, len(X_test)), random_state=RANDOM_STATE)

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_sample)

print('SHAP values shape:', shap_values.values.shape)

## 3. Global Feature Importance

This bar plot ranks features by their average impact on model output (magnitude only,
no direction). This is the plot to screenshot for a quick 'what matters most' summary.

In [ ]:
shap.plots.bar(shap_values, max_display=15)

## 4. Feature Effect Direction (Beeswarm Plot)

This shows not just which features matter, but HOW — e.g. does high BMI push predictions
toward higher risk (positive SHAP value) or lower risk? Each dot is one patient in the sample;
color shows the feature's value (red = high, blue = low).

In [ ]:
shap.plots.beeswarm(shap_values, max_display=15)

## 5. Individual Prediction Explanation (Waterfall Plot)

This explains a single prediction — useful for showing 'here's why the model flagged this
specific person as high-risk.' Great for a portfolio screenshot showing real-world applicability.

In [ ]:
# Pick an example the model is confident is high-risk
probs = xgb_model.predict_proba(X_sample)[:, 1]
high_risk_idx = np.argmax(probs)

print(f'Predicted probability of diabetes/prediabetes: {probs[high_risk_idx]:.3f}')
shap.plots.waterfall(shap_values[high_risk_idx], max_display=15)

In [ ]:
# And a low-risk example for contrast
low_risk_idx = np.argmin(probs)

print(f'Predicted probability of diabetes/prediabetes: {probs[low_risk_idx]:.3f}')
shap.plots.waterfall(shap_values[low_risk_idx], max_display=15)

## 6. Dependence Plots for Top Features

Shows how a single feature's value relates to its SHAP impact, and whether it interacts
with another feature (shown via color).

In [ ]:
# Update this list based on what showed up as top features in the plots above
top_features = ['BMI', 'GenHlth', 'Age', 'HighBP']
top_features = [f for f in top_features if f in X_sample.columns]

for feature in top_features:
    shap.plots.scatter(shap_values[:, feature], color=shap_values)
    plt.show()

## 7. Key Takeaways

*(Fill this in — this is the section that makes your project feel like a real analysis rather
than a modeling exercise. Answer: What are the top 3-5 drivers of predicted diabetes risk?
Do they make clinical/common sense? Any surprising interactions?)*

- 
- 
- 

**Next step:** Build the Streamlit app (`app/app.py`) so users can input their own health
indicators and get a live risk prediction with a SHAP-based explanation of the result.